# Imports e Setup

In [ ]:
import sys
import os
import torch
import zuko
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time



In [ ]:
# Adiciona a pasta mini_project_1 como root
sys.path.append(os.path.abspath('..')) 

# Configurando dispositivo de processamento
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


Modelos

In [3]:
from src.models.cnf import CNF
from src.models.vector_field import VectorField
from src.models.ffjord import FFJORD, hutchinson_trace_estimator

def build_realnvp(dim, hidden_features=(64,64), n_transforms=6):
    return zuko.flows.coupling.RealNVP(
        features=dim,
        hidden_features=hidden_features,
        transforms=n_transforms
    )

# Milestone 3: FFJORD

## Fundamentação Teórica 

Para calcular o trace usada no cálculo de $logp(x)$, o CNF usa divergência exata, que precisa de integração numérica para computar o jacobiano. No FFJORD, usaremos um estimador para calcular o trace, diminuindo o custo computacional.

### Hutchinson's Trace Estimator

Para a matriz $A \in \R^{d \times d}$:
$$tr(A) = E_ \epsilon [\epsilon^TA_ \epsilon]$$
onde $\epsilon \sim p$ com $E[\epsilon] = 0$ e $E[\epsilon \epsilon^T] = I$

Distribuições comuns:
- Gaussian: $\epsilon \sim N(0,I)$
- Rademancher: $\epsilon_i \sim Uniform({-1,+1})$

## Treinamento

Preprocessamento

In [4]:
def preprocess_mnist(x):
    # x ∈ [0, 255]
    x = x + torch.rand_like(x)
    x = x / 256.0

    alpha = 0.05
    x = alpha + (1 - 2 * alpha) * x
    x = torch.logit(x)

    return x

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(preprocess_mnist)
])

train_ds = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_ds  = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=128)

In [5]:
def train_epoch(model, opt):
    model.train()
    total = 0
    for x, _ in train_loader:
        x = x.to(device)
        x = x.view(x.size(0), -1)
        loss = -model.log_prob(x).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    return total / len(train_loader)

In [6]:
epochs = 50
times = []

vf = VectorField(features=784).to(device)
ffjord = FFJORD(vf, device, n_trace_samples=1, noise='rademacher').to(device)
opt = torch.optim.Adam(ffjord.parameters(), lr=1e-4)

for ep in range(epochs):
    start = time.time()
    loss = train_epoch(ffjord, opt)
    times.append(time.time() - start)
    print(f"Epoch {ep+1}: Loss={loss:.2f}, Time={times[-1]:.2f}s")

c:\Users\loren\Projetos_Antigos\flow\.venv\Lib\site-packages\torch\autograd\graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1: Loss=1073.90, Time=1015.70s


KeyboardInterrupt: 

## Análise

Depois de ficar 863 minutos rodando, tendo levado 1015.70s pra rodar 1 época, concluímos que é inviável rodar o FFJORD nas condições atuais. Provavelmente, teríamos que ajustar algum parametro para sua execução ser viável.

### Variância do estimador

### Discussão

| Método | Dataset | Dim | Log-likelihood | NFE (train) | Time/epoch | Trace Cost|
|---|----|---|---|--|---|--|
|Real NVP         |MNIST|784||||N/A|
|CNF (trace exato)|2D   |2  ||||$O(d^2)$|
|FFJORD           |MNIST|784||||$O(d)$|